# Thai Call Center ASR - Colab Notebook

Self-contained notebook for Thai banking call-center ASR. Run top to bottom for a baseline, or rerun individual cells after errors. No helper `.py` files are required.

## 1. Project config + runtime check

In [11]:
import os, re, csv, json, time, math, wave, random, shutil, zipfile, subprocess, sys
from pathlib import Path
from datetime import datetime
from getpass import getpass

IS_COLAB = 'google.colab' in sys.modules
DRIVE_ROOT = Path('/content/drive/MyDrive/Thai-call-center-ASR')
DATA_DIR = DRIVE_ROOT / 'audio_final'
AUDIO_DIR = DATA_DIR / 'audio'
SAMPLE_CSV = DATA_DIR / 'sample_submission.csv'
ZIP_PATH = DRIVE_ROOT / 'individual-test-thai-call-center-asr.zip'
WORK_DIR = DRIVE_ROOT / 'work'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
SUBMISSION_DIR = WORK_DIR / 'submissions'
REPORT_DIR = WORK_DIR / 'reports'

OPENAI_MODEL = 'gpt-4o-transcribe'
WHISPER_MODEL = 'large-v3-turbo'
WHISPER_FALLBACK_MODEL = 'medium'
RANDOM_SEED = 42
SAVE_EVERY = 10

random.seed(RANDOM_SEED)
for p in [WORK_DIR, CHECKPOINT_DIR, SUBMISSION_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version)
print('Colab:', IS_COLAB)
try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except Exception as e:
    print('Torch check skipped:', e)
print('Drive root:', DRIVE_ROOT)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Colab: True
CUDA available: True
GPU: Tesla T4
Drive root: /content/drive/MyDrive/Thai-call-center-ASR


## 2. Mount Google Drive + locate dataset

Expected layout: `/content/drive/MyDrive/Thai-call-center-ASR/audio_final/audio/*.wav` and `sample_submission.csv`. If only the zip exists, this cell extracts it.

In [9]:
from pathlib import Path

zip_name = "individual-test-thai-call-center-asr.zip"

search_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/content/drive"),
]

found = []

for root in search_roots:
    if not root.exists():
        continue
    print("Searching in:", root)
    try:
        for p in root.rglob(zip_name):
            found.append(p)
            print("FOUND:", p)
    except Exception as e:
        print("Skipped:", root, repr(e))

print("\nTotal found:", len(found))

Searching in: /content/drive/MyDrive
FOUND: /content/drive/MyDrive/individual-test-thai-call-center-asr.zip
FOUND: /content/drive/MyDrive/Thai-call-center-ASR/individual-test-thai-call-center-asr.zip
Searching in: /content/drive
FOUND: /content/drive/MyDrive/individual-test-thai-call-center-asr.zip
FOUND: /content/drive/MyDrive/Thai-call-center-ASR/individual-test-thai-call-center-asr.zip

Total found: 4


## 3. Install dependencies

In [ ]:
# Rerunnable install cell. Colab may ask to restart runtime after dependency changes; usually this stack works without restart.
!pip -q install -U pandas numpy tqdm openai faster-whisper jiwer python-Levenshtein soundfile librosa scipy fastapi uvicorn nest-asyncio python-multipart

## 4. Load sample submission

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

sample_df = pd.read_csv(SAMPLE_CSV)
assert {'file_name', 'text'}.issubset(sample_df.columns), sample_df.columns.tolist()
sample_df['file_name'] = sample_df['file_name'].astype(str)
print(sample_df.shape)
display(sample_df.head())

## 5. Audit filenames, variants, groups, durations

In [ ]:
VARIANT_SUFFIXES = ['noise', 'phone', 'fast', 'slow', 'pitch']
VARIANT_RE = re.compile(r'_(noise|phone|fast|slow|pitch)\.wav$', re.IGNORECASE)

def parse_audio_name(file_name):
    variant = 'original'
    group_id = file_name
    m = VARIANT_RE.search(file_name)
    if m:
        variant = m.group(1).lower()
        group_id = VARIANT_RE.sub('.wav', file_name)
    return group_id, variant

def safe_wav_duration(path):
    try:
        with wave.open(str(path), 'rb') as w:
            return w.getnframes() / float(w.getframerate()), w.getframerate()
    except Exception:
        return np.nan, np.nan

audit_df = sample_df[['file_name']].copy()
audit_df[['group_id', 'variant']] = audit_df['file_name'].apply(lambda x: pd.Series(parse_audio_name(x)))
group_sizes = audit_df.groupby('group_id')['file_name'].transform('count')
audit_df['group_size'] = group_sizes
audit_df['group_type'] = np.where(audit_df['group_size'].eq(6), 'complete_6', np.where(audit_df['group_size'].eq(1), 'singleton', 'partial'))
audit_df['audio_path'] = audit_df['file_name'].apply(lambda x: str(AUDIO_DIR / x))
audit_df['exists'] = audit_df['audio_path'].apply(lambda x: Path(x).exists())

# Duration audit can be slow. Start with all rows; reduce MAX_DURATION_ROWS if needed.
MAX_DURATION_ROWS = len(audit_df)
durations, sample_rates = [], []
for p in tqdm(audit_df['audio_path'].head(MAX_DURATION_ROWS), desc='duration audit'):
    d, sr = safe_wav_duration(Path(p))
    durations.append(d); sample_rates.append(sr)
if MAX_DURATION_ROWS < len(audit_df):
    durations += [np.nan] * (len(audit_df) - MAX_DURATION_ROWS)
    sample_rates += [np.nan] * (len(audit_df) - MAX_DURATION_ROWS)
audit_df['duration_sec'] = durations
audit_df['sample_rate'] = sample_rates

audit_summary = pd.DataFrame({
    'metric': ['rows', 'unique_groups', 'complete_6_groups', 'singleton_groups', 'partial_groups', 'missing_audio', 'avg_duration_sec', 'median_duration_sec', 'total_audio_min'],
    'value': [
        len(audit_df),
        audit_df['group_id'].nunique(),
        audit_df.loc[audit_df['group_type'].eq('complete_6'), 'group_id'].nunique(),
        audit_df.loc[audit_df['group_type'].eq('singleton'), 'group_id'].nunique(),
        audit_df.loc[audit_df['group_type'].eq('partial'), 'group_id'].nunique(),
        int((~audit_df['exists']).sum()),
        float(audit_df['duration_sec'].mean()),
        float(audit_df['duration_sec'].median()),
        float(audit_df['duration_sec'].sum() / 60.0),
    ]
})
audit_summary.to_csv(WORK_DIR / 'audit_summary.csv', index=False, encoding='utf-8-sig')
display(audit_summary)
display(audit_df.groupby(['group_type','variant']).size().reset_index(name='n'))

## 6. Define Thai text normalization

In [ ]:
ZERO_WIDTH_RE = re.compile('[\u200b\u200c\u200d\ufeff]')
PUNCT_RE = re.compile(r'[\.\,\?\!\:\;\"\'\“\”\‘\’\(\)\[\]\{\}\<\>\|/\\]+')
SPACE_RE = re.compile(r'\s+')

def normalize_thai_text(text, remove_punct=False, no_space=False):
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ''
    text = str(text)
    text = ZERO_WIDTH_RE.sub('', text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    if remove_punct:
        text = PUNCT_RE.sub('', text)
    text = SPACE_RE.sub(' ', text).strip()
    if no_space:
        text = text.replace(' ', '')
    return text

def looks_like_bad_transcript(text):
    t = normalize_thai_text(text, remove_punct=False)
    if not t:
        return True
    lowered = t.lower()
    bad_phrases = ['i cannot', 'i can\'t', 'sorry', 'ขออภัย', 'ไม่สามารถถอด', 'transcribe', 'translation']
    if any(p in lowered for p in bad_phrases):
        return True
    thai_chars = len(re.findall(r'[\u0E00-\u0E7F]', t))
    latin_chars = len(re.findall(r'[A-Za-z]', t))
    if latin_chars > thai_chars and latin_chars > 8:
        return True
    return False

FILLER_WORDS = ['อืม', 'เอ่อ', 'เออ', 'ค่ะ', 'คะ', 'ครับ', 'นะคะ', 'นะครับ']
DOMAIN_WORDS = ['บัตรเครดิต', 'สินเชื่อ', 'บัญชี', 'โมบาย', 'แบงก์กิ้ง', 'ธนาคาร', 'โอนเงิน', 'อายัด', 'ยอดค้าง']

print(normalize_thai_text('  เอ่อ... ขอ ยกเลิกบัตรเครดิต ค่ะ!!!  ', remove_punct=True))

## 7. Define OpenAI transcription client

In [ ]:
OPENAI_PROMPT = '''Transcribe Thai banking call center speech exactly.
Keep filler words and backchannels such as อืม, เอ่อ, ค่ะ, ครับ, คะ, นะคะ.
Do not summarize. Do not translate. Output Thai transcript only.
Common terms: บัตรเครดิต, สินเชื่อ, บัญชี, โมบายแบงก์กิ้ง, แอปธนาคาร, ยกเลิกบัตร, อายัดบัตร, โอนเงิน, ยอดค้างชำระ.'''

def get_openai_api_key():
    key = os.environ.get('OPENAI_API_KEY')
    if key:
        return key
    if IS_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get('OPENAI_API_KEY')
            if key:
                os.environ['OPENAI_API_KEY'] = key
                return key
        except Exception as e:
            print('Colab Secrets lookup skipped:', e)
    key = getpass('Paste OPENAI_API_KEY for this runtime only: ')
    os.environ['OPENAI_API_KEY'] = key
    return key

def make_openai_client():
    from openai import OpenAI
    get_openai_api_key()
    return OpenAI()

def openai_transcribe_one(client, audio_path, model=OPENAI_MODEL, prompt=OPENAI_PROMPT, retries=3):
    last_error = ''
    for attempt in range(retries):
        try:
            with open(audio_path, 'rb') as f:
                result = client.audio.transcriptions.create(
                    model=model,
                    file=f,
                    language='th',
                    prompt=prompt,
                    response_format='text',
                )
            return str(result), ''
        except Exception as e:
            last_error = repr(e)
            sleep_s = min(30, 2 ** attempt)
            print(f'OpenAI error attempt {attempt+1}/{retries}: {last_error}; sleeping {sleep_s}s')
            time.sleep(sleep_s)
    return '', last_error

print('OpenAI helper ready. Client is created only when you run an OpenAI pass.')

## 8. Define faster-whisper transcription

In [ ]:
def load_faster_whisper(model_name=WHISPER_MODEL):
    from faster_whisper import WhisperModel
    try:
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    except Exception:
        device = 'cpu'
    compute_type = 'float16' if device == 'cuda' else 'int8'
    print('Loading faster-whisper:', model_name, device, compute_type)
    try:
        return WhisperModel(model_name, device=device, compute_type=compute_type)
    except Exception as e:
        print('Primary Whisper load failed:', repr(e))
        print('Trying fallback:', WHISPER_FALLBACK_MODEL)
        return WhisperModel(WHISPER_FALLBACK_MODEL, device=device, compute_type=compute_type)

def whisper_transcribe_one(model, audio_path, beam_size=5, use_vad=True):
    try:
        segments, info = model.transcribe(
            str(audio_path),
            language='th',
            task='transcribe',
            beam_size=beam_size,
            temperature=0.0,
            vad_filter=use_vad,
            condition_on_previous_text=False,
        )
        text = ''.join(seg.text for seg in segments)
        return text, ''
    except Exception as e:
        return '', repr(e)

print('faster-whisper helpers ready. Model loads only when you run a Whisper pass.')

## Shared checkpoint and target helpers

In [ ]:
CHECKPOINT_COLUMNS = ['file_name','group_id','variant','model','prompt_version','audio_source','text_raw','text_norm','duration_sec','runtime_sec','status','error','created_at']

def load_checkpoint(path):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        for c in CHECKPOINT_COLUMNS:
            if c not in df.columns:
                df[c] = ''
        return df[CHECKPOINT_COLUMNS]
    return pd.DataFrame(columns=CHECKPOINT_COLUMNS)

def save_checkpoint(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df[CHECKPOINT_COLUMNS].to_csv(path, index=False, encoding='utf-8-sig')

def completed_file_set(df, model_name, audio_source):
    if df.empty:
        return set()
    ok = df['status'].eq('ok') & df['model'].eq(model_name) & df['audio_source'].eq(audio_source)
    return set(df.loc[ok, 'file_name'].astype(str))

def make_checkpoint_row(file_name, model_name, prompt_version, audio_source, text_raw, error, runtime_sec):
    group_id, variant = parse_audio_name(file_name)
    duration_sec = audit_df.set_index('file_name').get('duration_sec', pd.Series()).get(file_name, np.nan)
    status = 'ok' if text_raw and not error else 'error'
    return {
        'file_name': file_name,
        'group_id': group_id,
        'variant': variant,
        'model': model_name,
        'prompt_version': prompt_version,
        'audio_source': audio_source,
        'text_raw': text_raw,
        'text_norm': normalize_thai_text(text_raw, remove_punct=False),
        'duration_sec': duration_sec,
        'runtime_sec': runtime_sec,
        'status': status,
        'error': error,
        'created_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    }

def choose_files_by_variants(df, variants=('original','slow'), include_singletons=True):
    mask = df['variant'].isin(variants)
    if include_singletons:
        mask = mask | df['group_type'].eq('singleton')
    return df.loc[mask & df['exists'], 'file_name'].drop_duplicates().tolist()

def make_pilot_files(n_groups=50, n_singletons=25):
    complete_groups = sorted(audit_df.loc[audit_df['group_type'].eq('complete_6'), 'group_id'].unique())
    singleton_files = sorted(audit_df.loc[audit_df['group_type'].eq('singleton'), 'file_name'].unique())
    rng = random.Random(RANDOM_SEED)
    pick_groups = rng.sample(complete_groups, min(n_groups, len(complete_groups)))
    group_files = audit_df.loc[audit_df['group_id'].isin(pick_groups) & audit_df['variant'].isin(['original','slow']), 'file_name'].tolist()
    pick_singletons = rng.sample(singleton_files, min(n_singletons, len(singleton_files)))
    return group_files + pick_singletons

PILOT_FILES = make_pilot_files()
MAIN_BASELINE_FILES = choose_files_by_variants(audit_df, variants=('original','slow'), include_singletons=True)
HARD_VARIANT_FILES = choose_files_by_variants(audit_df, variants=('phone','noise','fast'), include_singletons=False)
print('pilot files:', len(PILOT_FILES))
print('main baseline files:', len(MAIN_BASELINE_FILES))
print('hard variant files:', len(HARD_VARIANT_FILES))

## 9. Run pilot subset

In [ ]:
RUN_OPENAI_PILOT = False  # set True when ready to spend API budget
RUN_WHISPER_PILOT = False # set True after GPU runtime is ready

def run_openai_pass(file_names, checkpoint_name='openai_pilot.csv', audio_source='raw', prompt_version='v1'):
    cp_path = CHECKPOINT_DIR / checkpoint_name
    cp = load_checkpoint(cp_path)
    done = completed_file_set(cp, OPENAI_MODEL, audio_source)
    client = make_openai_client()
    rows = []
    for i, fn in enumerate(tqdm(file_names, desc=f'OpenAI {checkpoint_name}')):
        if fn in done:
            continue
        start = time.time()
        text, err = openai_transcribe_one(client, AUDIO_DIR / fn)
        rows.append(make_checkpoint_row(fn, OPENAI_MODEL, prompt_version, audio_source, text, err, time.time() - start))
        if len(rows) % SAVE_EVERY == 0:
            cp = pd.concat([cp, pd.DataFrame(rows)], ignore_index=True)
            save_checkpoint(cp, cp_path)
            rows = []
    if rows:
        cp = pd.concat([cp, pd.DataFrame(rows)], ignore_index=True)
        save_checkpoint(cp, cp_path)
    return load_checkpoint(cp_path)

def run_whisper_pass(file_names, checkpoint_name='whisper_pilot.csv', audio_source='raw'):
    cp_path = CHECKPOINT_DIR / checkpoint_name
    cp = load_checkpoint(cp_path)
    done = completed_file_set(cp, WHISPER_MODEL, audio_source)
    model = load_faster_whisper(WHISPER_MODEL)
    rows = []
    for i, fn in enumerate(tqdm(file_names, desc=f'Whisper {checkpoint_name}')):
        if fn in done:
            continue
        start = time.time()
        text, err = whisper_transcribe_one(model, AUDIO_DIR / fn)
        rows.append(make_checkpoint_row(fn, WHISPER_MODEL, 'none', audio_source, text, err, time.time() - start))
        if len(rows) % SAVE_EVERY == 0:
            cp = pd.concat([cp, pd.DataFrame(rows)], ignore_index=True)
            save_checkpoint(cp, cp_path)
            rows = []
    if rows:
        cp = pd.concat([cp, pd.DataFrame(rows)], ignore_index=True)
        save_checkpoint(cp, cp_path)
    return load_checkpoint(cp_path)

pilot_outputs = []
if RUN_OPENAI_PILOT:
    pilot_outputs.append(run_openai_pass(PILOT_FILES, 'openai_pilot.csv'))
if RUN_WHISPER_PILOT:
    pilot_outputs.append(run_whisper_pass(PILOT_FILES, 'whisper_pilot.csv'))
print('Pilot flags:', RUN_OPENAI_PILOT, RUN_WHISPER_PILOT)

## 10. Inspect pilot outputs

In [ ]:
pilot_frames = []
for name in ['openai_pilot.csv', 'whisper_pilot.csv']:
    path = CHECKPOINT_DIR / name
    if path.exists():
        pilot_frames.append(pd.read_csv(path))
if pilot_frames:
    pilot_df = pd.concat(pilot_frames, ignore_index=True)
    display(pilot_df[['file_name','variant','model','status','text_norm','error']].sample(min(30, len(pilot_df)), random_state=RANDOM_SEED))
    display(pilot_df.groupby(['model','variant','status']).size().reset_index(name='n'))
else:
    print('No pilot checkpoints yet. Set pilot flags in the previous cell and run it.')

## 11. Run OpenAI main pass with checkpoint

In [ ]:
RUN_OPENAI_MAIN = False  # set True after pilot looks good
if RUN_OPENAI_MAIN:
    openai_main_df = run_openai_pass(MAIN_BASELINE_FILES, 'openai_main_original_slow_singletons.csv')
    display(openai_main_df.tail())
else:
    print('Set RUN_OPENAI_MAIN=True to run OpenAI original+slow+singletons pass.')

## 12. Run Whisper open-weight pass with checkpoint

In [ ]:
RUN_WHISPER_MAIN = False  # set True on Colab GPU
if RUN_WHISPER_MAIN:
    whisper_main_df = run_whisper_pass(MAIN_BASELINE_FILES, 'whisper_main_original_slow_singletons.csv')
    display(whisper_main_df.tail())
else:
    print('Set RUN_WHISPER_MAIN=True to run faster-whisper original+slow+singletons pass.')

## 13. Optional noise-specific preprocessing candidate

In [ ]:
RUN_NOISE_PREPROCESS_PASS = False
NOISE_PREP_DIR = WORK_DIR / 'noise_preprocessed'
NOISE_PREP_DIR.mkdir(parents=True, exist_ok=True)

def preprocess_noise_audio(in_path, out_path):
    import librosa, soundfile as sf
    from scipy.signal import butter, sosfilt
    y, sr = librosa.load(str(in_path), sr=None, mono=True)
    low = max(80, 120) / (sr / 2)
    high = min(3800, sr / 2 - 100) / (sr / 2)
    if 0 < low < high < 1:
        sos = butter(4, [low, high], btype='bandpass', output='sos')
        y = sosfilt(sos, y)
    peak = np.max(np.abs(y)) if len(y) else 0
    if peak > 0:
        y = 0.95 * y / peak
    sf.write(str(out_path), y, sr)
    return out_path

def run_whisper_noise_preprocessed(file_names):
    cp_path = CHECKPOINT_DIR / 'whisper_noise_preprocessed.csv'
    cp = load_checkpoint(cp_path)
    done = completed_file_set(cp, WHISPER_MODEL, 'noise_preprocessed')
    model = load_faster_whisper(WHISPER_MODEL)
    rows = []
    noise_files = [fn for fn in file_names if parse_audio_name(fn)[1] == 'noise']
    for fn in tqdm(noise_files, desc='Whisper noise preprocessed'):
        if fn in done:
            continue
        out_path = NOISE_PREP_DIR / fn
        start = time.time()
        try:
            preprocess_noise_audio(AUDIO_DIR / fn, out_path)
            text, err = whisper_transcribe_one(model, out_path)
        except Exception as e:
            text, err = '', repr(e)
        rows.append(make_checkpoint_row(fn, WHISPER_MODEL, 'none', 'noise_preprocessed', text, err, time.time() - start))
        if len(rows) % SAVE_EVERY == 0:
            cp = pd.concat([cp, pd.DataFrame(rows)], ignore_index=True)
            save_checkpoint(cp, cp_path)
            rows = []
    if rows:
        cp = pd.concat([cp, pd.DataFrame(rows)], ignore_index=True)
        save_checkpoint(cp, cp_path)
    return load_checkpoint(cp_path)

if RUN_NOISE_PREPROCESS_PASS:
    noise_df = run_whisper_noise_preprocessed(HARD_VARIANT_FILES)
    display(noise_df.tail())
else:
    print('Optional. Set RUN_NOISE_PREPROCESS_PASS=True after baseline passes.')

## 14. Consensus per group/singleton

In [ ]:
try:
    import Levenshtein
    def edit_distance(a, b): return Levenshtein.distance(a, b)
except Exception:
    from difflib import SequenceMatcher
    def edit_distance(a, b):
        return int(round((1 - SequenceMatcher(None, a, b).ratio()) * max(len(a), len(b), 1)))

def candidate_weight(row):
    model = str(row.get('model','')).lower()
    variant = str(row.get('variant',''))
    audio_source = str(row.get('audio_source',''))
    if audio_source == 'noise_preprocessed':
        return 0.80
    if 'gpt-4o' in model and variant == 'original':
        return 1.25
    if 'gpt-4o' in model and variant == 'slow':
        return 1.20
    if 'whisper' in model and variant == 'original':
        return 1.10
    if 'whisper' in model and variant == 'slow':
        return 1.05
    if variant in ['phone','noise','fast','pitch']:
        return 0.90
    return 1.0

def candidate_penalty(text, group_texts=None):
    if looks_like_bad_transcript(text):
        return 1e6
    penalty = 0.0
    norm = normalize_thai_text(text, remove_punct=False)
    if group_texts:
        lengths = [len(t) for t in group_texts if t]
        if lengths:
            med = float(np.median(lengths))
            if med > 0 and len(norm) < 0.45 * med:
                penalty += med
    punct_count = len(PUNCT_RE.findall(norm))
    penalty += punct_count * 1.5
    if any(w in norm for w in FILLER_WORDS):
        penalty -= 2.0
    if any(w in norm for w in DOMAIN_WORDS):
        penalty -= 1.0
    return penalty

def choose_consensus(group_candidates):
    ok = group_candidates[group_candidates['status'].eq('ok')].copy()
    ok['text_candidate'] = ok['text_raw'].apply(lambda x: normalize_thai_text(x, remove_punct=False))
    ok = ok[~ok['text_candidate'].apply(looks_like_bad_transcript)]
    if ok.empty:
        return '', 'no_valid_candidate', None
    texts = ok['text_candidate'].tolist()
    scores = []
    for idx, row in ok.iterrows():
        t = row['text_candidate']
        weighted_dist, total_w = 0.0, 0.0
        for jdx, other in ok.iterrows():
            if idx == jdx:
                continue
            w = candidate_weight(other)
            weighted_dist += w * edit_distance(t, other['text_candidate'])
            total_w += w
        avg_dist = weighted_dist / max(total_w, 1e-9)
        score = avg_dist + candidate_penalty(t, texts) - candidate_weight(row)
        scores.append((score, idx))
    scores.sort(key=lambda x: x[0])
    best_idx = scores[0][1]
    best = ok.loc[best_idx]
    return best['text_candidate'], 'ok', best.to_dict()

def load_all_candidate_checkpoints():
    frames = []
    for p in sorted(CHECKPOINT_DIR.glob('*.csv')):
        try:
            df = pd.read_csv(p)
            df['checkpoint_file'] = p.name
            frames.append(df)
        except Exception as e:
            print('Could not read checkpoint:', p, e)
    if not frames:
        return pd.DataFrame(columns=CHECKPOINT_COLUMNS + ['checkpoint_file'])
    return pd.concat(frames, ignore_index=True)

candidates_df = load_all_candidate_checkpoints()
if not candidates_df.empty:
    candidates_df.to_csv(WORK_DIR / 'experiment_candidates.csv', index=False, encoding='utf-8-sig')
print('candidate rows:', len(candidates_df))
display(candidates_df.groupby(['checkpoint_file','status']).size().reset_index(name='n') if not candidates_df.empty else candidates_df)

## 15. Generate submission CSV

In [ ]:
def build_submission(candidates_df, remove_punct=False, no_space=False, out_name='submission_keep_space.csv'):
    group_rows = []
    chosen_by_group = {}
    if candidates_df.empty:
        raise ValueError('No candidate checkpoints found. Run OpenAI or Whisper passes first.')
    for group_id, group_meta in tqdm(audit_df.groupby('group_id'), desc='consensus groups'):
        group_candidates = candidates_df[candidates_df['group_id'].eq(group_id)].copy()
        text, status, best = choose_consensus(group_candidates)
        final_text = normalize_thai_text(text, remove_punct=remove_punct, no_space=no_space)
        chosen_by_group[group_id] = final_text
        group_rows.append({
            'group_id': group_id,
            'group_size': int(group_meta['group_size'].iloc[0]),
            'group_type': group_meta['group_type'].iloc[0],
            'status': status,
            'chosen_text': final_text,
            'best_file_name': '' if best is None else best.get('file_name',''),
            'best_model': '' if best is None else best.get('model',''),
            'best_variant': '' if best is None else best.get('variant',''),
            'best_audio_source': '' if best is None else best.get('audio_source',''),
            'candidate_count': int(len(group_candidates)),
        })
    debug_df = pd.DataFrame(group_rows)
    debug_df.to_csv(WORK_DIR / 'group_consensus_debug.csv', index=False, encoding='utf-8-sig')
    out = sample_df[['file_name']].copy()
    out['group_id'] = out['file_name'].apply(lambda x: parse_audio_name(x)[0])
    out['text'] = out['group_id'].map(chosen_by_group).fillna('')
    out = out[['file_name','text']]
    out_path = SUBMISSION_DIR / out_name
    out.to_csv(out_path, index=False, encoding='utf-8-sig')
    return out, debug_df, out_path

if not candidates_df.empty:
    sub_keep, debug_keep, path_keep = build_submission(candidates_df, remove_punct=False, no_space=False, out_name='submission_keep_space.csv')
    sub_nopunct, _, path_nopunct = build_submission(candidates_df, remove_punct=True, no_space=False, out_name='submission_no_punct_keep_space.csv')
    sub_nospace, _, path_nospace = build_submission(candidates_df, remove_punct=True, no_space=True, out_name='submission_no_space_probe.csv')
    print('Saved:', path_keep)
    print('Saved:', path_nopunct)
    print('Saved:', path_nospace)
    display(sub_keep.head())
else:
    print('No candidates yet. Run transcription passes first.')

## 16. Export experiment report tables and validation checks

In [ ]:
def validate_submission(path):
    sub = pd.read_csv(path)
    checks = {
        'row_count_matches': len(sub) == len(sample_df),
        'file_order_matches': sub['file_name'].astype(str).tolist() == sample_df['file_name'].astype(str).tolist(),
        'has_columns': sub.columns.tolist() == ['file_name','text'],
        'no_null_text': sub['text'].notna().all(),
        'all_text_stringable': sub['text'].map(lambda x: isinstance(str(x), str)).all(),
    }
    sub_audit = sub.copy()
    sub_audit[['group_id','variant']] = sub_audit['file_name'].apply(lambda x: pd.Series(parse_audio_name(x)))
    grouped_unique_text = sub_audit.groupby('group_id')['text'].nunique(dropna=False)
    checks['grouped_files_share_text'] = bool((grouped_unique_text <= 1).all())
    result = pd.DataFrame([{'check': k, 'passed': bool(v)} for k, v in checks.items()])
    return result

validation_frames = []
for p in sorted(SUBMISSION_DIR.glob('submission_*.csv')):
    vf = validate_submission(p)
    vf.insert(0, 'submission', p.name)
    validation_frames.append(vf)
if validation_frames:
    validation_df = pd.concat(validation_frames, ignore_index=True)
    validation_df.to_csv(WORK_DIR / 'submission_validation.csv', index=False, encoding='utf-8-sig')
    display(validation_df)
else:
    print('No submissions found yet.')

if not candidates_df.empty:
    report = candidates_df.groupby(['model','variant','audio_source','status']).agg(
        n=('file_name','count'),
        avg_runtime_sec=('runtime_sec','mean'),
        avg_text_len=('text_norm', lambda s: np.mean([len(str(x)) for x in s]))
    ).reset_index()
    report.to_csv(WORK_DIR / 'experiment_report.csv', index=False, encoding='utf-8-sig')
    display(report)

## 17. Optional FastAPI demo cell for portfolio only

This is not needed for competition submission. Use it to demonstrate production-minded model serving in an interview.

In [ ]:
RUN_FASTAPI_DEMO = False
if RUN_FASTAPI_DEMO:
    import nest_asyncio, uvicorn
    from fastapi import FastAPI, UploadFile, File
    import tempfile
    nest_asyncio.apply()
    app = FastAPI(title='Thai Call Center ASR Demo')
    demo_model = load_faster_whisper(WHISPER_FALLBACK_MODEL)

    @app.post('/transcribe')
    async def transcribe(file: UploadFile = File(...)):
        suffix = Path(file.filename).suffix or '.wav'
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
            tmp.write(await file.read())
            tmp_path = tmp.name
        start = time.time()
        text, err = whisper_transcribe_one(demo_model, tmp_path)
        try:
            os.remove(tmp_path)
        except OSError:
            pass
        return {'text': normalize_thai_text(text), 'error': err, 'duration_ms': int((time.time() - start) * 1000)}

    uvicorn.run(app, host='0.0.0.0', port=8000)
else:
    print('FastAPI demo disabled. Set RUN_FASTAPI_DEMO=True only when you want to demo serving.')